In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from parse_transit_lines import parse_transit_lines

In [2]:
# extract transit links from the transit lines file
transit_links_df = parse_transit_lines()
display(transit_links_df.head())

e:\GitHub\TM1_Base_Network\utils\parse_transit_lines.py:70: UserWarning: LINE '21_5OUTA': missing ONEWAY; using F
  append_links()


,LINE NAME,MODE,ONEWAY,A,B,source
0,30_7AC,30,F,2412,2407,M:\Application\Model One\Networks\TM1_2015_Bas...
1,30_7AC,30,F,2407,2334,M:\Application\Model One\Networks\TM1_2015_Bas...
2,30_7AC,30,F,2334,2401,M:\Application\Model One\Networks\TM1_2015_Bas...
3,30_7AC,30,F,2401,12198,M:\Application\Model One\Networks\TM1_2015_Bas...
4,30_7AC,30,F,12198,11545,M:\Application\Model One\Networks\TM1_2015_Bas...


In [3]:
# drop rail and ferry links as they do not use roadway links
print('unique transit modes:', transit_links_df['MODE'].unique())

unique transit modes: [ 30  85  84  83  19  55 131 132 133 120 121  14  86  42  82  12  52 100
 101 102 103 104  87  70  16  20  21 110  93  95  60  68  24  66  81  28
  27  15  63  13  44  38  56  91  49  10  46  90  17  33 135 111  92 130
  18  88 107]


In [4]:
localbus_links_df = transit_links_df.loc[transit_links_df['MODE'] <= 78]
print('unique transit modes for local bus links:', localbus_links_df['MODE'].unique())
print('total local bus links, rows:', len(localbus_links_df))

expressbus_links_df = transit_links_df.loc[(transit_links_df['MODE'] >= 80) & (transit_links_df['MODE'] < 100)]
print('unique transit modes for express bus links:', expressbus_links_df['MODE'].unique())
print('total express bus links, rows:', len(expressbus_links_df))

unique transit modes for local bus links: [30 19 55 14 42 12 52 70 16 20 21 60 68 24 66 28 27 15 63 13 44 38 56 49
 10 46 17 33 18]
total local bus links, rows: 22052
unique transit modes for express bus links: [85 84 83 86 82 87 93 95 81 91 90 92 88]
total express bus links, rows: 5643


In [5]:
roadway_links_df = gpd.read_file(
    Path.cwd().parent / "shapefiles" / "network_links.shp"
)
print('loaded roadway links, rows:', len(roadway_links_df))
roadway_links_df = roadway_links_df.loc[roadway_links_df['FT'] != 6][['A', 'B', 'FT', 'DISTANCE', 'SPDCLASS', 'CAPCLASS', 'LANES', 'USE']]
print('filtered roadway links (with FT != 6), rows:', len(roadway_links_df))
print('unique A-B pairs:', roadway_links_df[['A', 'B']].drop_duplicates().shape[0])
display(roadway_links_df[['A', 'B', 'DISTANCE', 'SPDCLASS', 'CAPCLASS', 'LANES', 'USE']].head())

c:\Users\ywang\AppData\Local\anaconda3\envs\rawfood\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


loaded roadway links, rows: 33953
filtered roadway links (with FT != 6), rows: 25096
unique A-B pairs: 25096


,A,B,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE
2703,1601,8838,0.29,47,47,2,1
2704,1601,8844,0.29,47,47,2,1
2706,1602,11993,0.81,29,29,4,1
2707,1603,1604,0.49,45,45,1,1
2709,1603,11944,0.59,29,29,4,1


In [6]:
localbus_compare = localbus_links_df.merge(
    roadway_links_df,
    how='left',
    left_on=['A', 'B'],
    right_on=['A', 'B'],
    suffixes=('_transit', '_roadway'),
    indicator=True
)
# compare_links
print('total compared links, rows:', len(localbus_compare))
# print('unique A-B pairs in compared links:', localbus_compare[['A', 'B']].drop_duplicates().shape[0])

total compared links, rows: 22052


In [7]:
localbus_compare.loc[localbus_compare['_merge'] == 'left_only']

,LINE NAME,MODE,ONEWAY,A,B,source,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE,_merge
658,30_76AC,30,F,2412,2336,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1992,30_805,30,F,3167,2938,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2291,30_200SB,30,T,3886,9433,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2293,30_200SB,30,T,20243,3898,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2542,30_801AC,30,F,2818,2883,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21546,70_GG57SB,70,T,9650,7907,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21565,70_GG57NB,70,T,7908,9440,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21578,70_GG57NB,70,T,9627,7971,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21959,44_709,44,T,1651,20217,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [8]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')]

,LINE NAME,MODE,ONEWAY,A,B,source,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE,_merge
658,30_76AC,30,F,2412,2336,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1992,30_805,30,F,3167,2938,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2542,30_801AC,30,F,2818,2883,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2543,30_801AC,30,F,2883,2884,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2544,30_801AC,30,F,2884,2844,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21364,24_278,24,F,6057,6059,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21365,24_278,24,F,6059,6053,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21366,24_278,24,F,6053,6054,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21396,24_142,24,F,6504,6439,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [21]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')][['A', 'B']].drop_duplicates().shape[0]

158

In [9]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'F')]['source'].unique()

<ArrowStringArray>
[                              'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\AC_Local.tpl',
                            'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\CCCTA_local.tpl',
                              'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\GGT_Local.tpl',
                         'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\Muni_CableCars.tpl',
                             'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\MUNI_Local.tpl',
                            'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\SCVTA_Local.tpl',
                             'E:\GitHub\NetworkWrangler\scripts\scratch\Wrangler_tmp_2026Mar31.132925\VTA_Next\VTA.lin',
      'E:\GitHub\NetworkWrangler\scripts\scratch\Wrangler_tmp_2026Mar31.132925\GTFS2023_SF_Muni\SF_2023_new_routes.lin',
       'E:\Gi

In [11]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')][['A', 'B']].drop_duplicates().shape[0]

164

In [10]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')]['source'].unique()

<ArrowStringArray>
[                                    'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\AC_Local.tpl',
                             'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\CALTRAIN_Shuttle.tpl',
                                    'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\Fairfield.tpl',
                                    'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\GGT_Local.tpl',
                                   'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\MUNI_Local.tpl',
                               'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\SamTrans_Local.tpl',
                                'M:\Application\Model One\Networks\TM1_2015_Base_Network\trn\transit_lines\SCVTA_Shuttle.TPL',
                                     'M:\Application\Model One\Networks\TM1_2015_Base_Networ

In [12]:
localbus_compare.loc[(localbus_compare['_merge'] == 'left_only') & (localbus_compare['ONEWAY'] == 'T')]

,LINE NAME,MODE,ONEWAY,A,B,source,FT,DISTANCE,SPDCLASS,CAPCLASS,LANES,USE,_merge
2291,30_200SB,30,T,3886,9433,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2293,30_200SB,30,T,20243,3898,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3279,14_CTBWAM,14,T,5456,10135,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3281,14_CTBWAM,14,T,20170,5457,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3296,14_CTBWPM,14,T,5457,20170,M:\Application\Model One\Networks\TM1_2015_Bas...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21546,70_GG57SB,70,T,9650,7907,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21565,70_GG57NB,70,T,7908,9440,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21578,70_GG57NB,70,T,9627,7971,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
21959,44_709,44,T,1651,20217,E:\GitHub\NetworkWrangler\scripts\scratch\Wran...,NaN,NaN,NaN,NaN,NaN,NaN,left_only
